# 22.1 自定义算子与 Kernel 工程

演示 RMSNorm 分解、RoPE 参考实现、回退代价估算与数值对齐门禁。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
assert torch is not None


def rmsnorm(x, w, eps=1e-6):
    # 可分解为：pow -> mean -> add -> rsqrt -> mul -> mul
    rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    return x * rms * w


class Rope:
    def __init__(self, dim, base=10000):
        inv = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register = inv

    def __call__(self, x, pos):
        # x: [B,T,D]
        half = x.shape[-1] // 2
        inv = self.register.to(x.device)[:half]
        t = pos.float().unsqueeze(-1) * inv
        cos, sin = t.cos(), t.sin()
        x1, x2 = x[..., :half], x[..., half:]
        out1 = x1 * cos - x2 * sin
        out2 = x1 * sin + x2 * cos
        return torch.cat([out1, out2], dim=-1)


x = torch.randn(2, 8, 64)
w = torch.ones(64)
y = rmsnorm(x, w)
rope = Rope(64)
yp = rope(y, torch.arange(8))
print("rmsnorm+rope", tuple(yp.shape))


def fallback_cost(npu_ms, cpu_ms, fallback_ratio):
    return npu_ms * (1 - fallback_ratio) + cpu_ms * fallback_ratio

print("latency if 5% cpu fallback:", fallback_cost(10, 80, 0.05), "ms")


def parity_ok(ref, test, cos_th=0.999, maxerr=1e-2):
    ref = ref.flatten().float(); test = test.flatten().float()
    cos = torch.nn.functional.cosine_similarity(ref, test, dim=0).item()
    err = (ref - test).abs().max().item()
    return cos >= cos_th and err <= maxerr, {"cos": cos, "maxerr": err}

ok, st = parity_ok(y, y * 1.0001)
print("parity", ok, st)

## 小结

先分解、再改模型、再写 Kernel；任何自定义 op 必须有对齐门禁，并计入回退代价。